# BinarySearch (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/BinarySearch_cpp.ipynb)

Chapter 13 — Correctness. Runnable notebook of `BinarySearch`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/BinarySearch_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness/BinarySearch_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the C++ code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## Binary search: BinarySearch

**Input:** an array $a[1..n]$ sorted in non-decreasing order and a value $m$. **Output:** an index $k$ with $a[k] = m$ if $m$ is in $a$, $-1$ otherwise. **Invariant:** (I1) $1 \le left$ and $right \le n$; (I2) if $m$ is in $a$, then $m$ is in $a[left..right]$.

Pseudo-code: section 13.4 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [1]:
%%writefile binary_search.cpp
#include <algorithm>
#include <cstdio>
#include <string>
using namespace std;

const int MAXN = 64;

// "[a[from], ..., a[to-1]]"
string Fmt(const int* a, int from, int to)
{
    string s = "[";
    for (int i = from; i < to; i++)
    {
        if (i > from) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

bool Equal(const int* x, const int* y, int len)     // x[0..len-1] == y[0..len-1]
{
    for (int k = 0; k < len; k++)
        if (x[k] != y[k]) return false;
    return true;
}

// Invariant (I1) 1 <= left and right <= n, (I2) if m is in a then it is in a[left..right]; mid = -1: empty interval
void PrintRow(const int* a, int n, int m, int left, int right, int mid)
{
    bool inA = false, inSub = false;
    for (int k = 0; k < n; k++)
    {
        if (a[k] == m)
        {
            inA = true;
            if (k >= left && k <= right) inSub = true;
        }
    }
    bool ok = (left + 1 >= 1 && right + 1 <= n && (!inA || inSub));
    char action[100];
    if (mid < 0)
        snprintf(action, sizeof action, "left > right => return -1");
    else if (a[mid] == m)
        snprintf(action, sizeof action, "mid = %d, a[%d] = %d = %d => return %d", mid + 1, mid + 1, a[mid], m, mid + 1);
    else if (a[mid] < m)
        snprintf(action, sizeof action, "mid = %d, a[%d] = %d < %d => left = %d", mid + 1, mid + 1, a[mid], m, mid + 2);
    else
        snprintf(action, sizeof action, "mid = %d, a[%d] = %d > %d => right = %d", mid + 1, mid + 1, a[mid], m, mid);
    printf("%4d | %5d | %-27s | %-11s | %s\n", left + 1, right + 1, Fmt(a, left, right + 1).c_str(), ok ? "ok" : "FALSE", action);
}

// BinarySearch(a, n, m)
int BinarySearch(int a[], int n, int m, bool trace = false)
{
    int left = 0;                                  // left := 1
    int right = n - 1;                             // right := n
    while (left <= right)
    {
        int mid = left + (right - left) / 2;
        if (trace) PrintRow(a, n, m, left, right, mid);   // invariant row for this evaluation
        if (a[mid] == m)
        {
            return mid;
        }
        if (a[mid] < m)
        {
            left = mid + 1;
        }
        else
        {
            right = mid - 1;
        }
    }
    if (trace) PrintRow(a, n, m, left, right, -1);
    return -1;
}

#ifndef SOLO_LIBRERIA
int main()
{
    int a[] = {3, 9, 11, 12, 12, 16, 19};          // example of the lecture notes
    int n = 7;
    printf("a = %s, n = %d\n", Fmt(a, 0, n).c_str(), n);

    int m = 10;
    printf("\n");
    printf("m = %d\n", m);
    printf("left | right | a[left..right]              | invariant   | action\n");
    int k = BinarySearch(a, n, m, true);
    printf("m = %d not found: it returns %d\n", m, k);

    m = 12;
    printf("\n");
    printf("m = %d\n", m);
    printf("left | right | a[left..right]              | invariant   | action\n");
    k = BinarySearch(a, n, m, true);
    printf("m = %d found: index %d in the code (position %d in the pseudo-code)\n", m, k, k + 1);

    // Exhaustive check for m = 0, ..., 20
    int errors = 0;
    for (m = 0; m <= 20; m++)
    {
        k = BinarySearch(a, n, m);
        bool inA = false;
        for (int t = 0; t < n; t++) if (a[t] == m) inA = true;
        if ((inA && (k < 0 || a[k] != m)) || (!inA && k != -1)) errors++;
    }
    printf("\n");
    printf("Check for m = 0, ..., 20: %d errors\n", errors);
    return 0;
}
#endif

Writing binary_search.cpp


In [2]:
!g++ -std=c++17 -O2 -o binary_search binary_search.cpp && ./binary_search

a = [3, 9, 11, 12, 12, 16, 19], n = 7

m = 10
left | right | a[left..right]              | invariant   | action
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 > 10 => right = 3
   1 |     3 | [3, 9, 11]                  | ok          | mid = 2, a[2] = 9 < 10 => left = 3
   3 |     3 | [11]                        | ok          | mid = 3, a[3] = 11 > 10 => right = 2
   3 |     2 | []                          | ok          | left > right => return -1
m = 10 not found: it returns -1

m = 12
left | right | a[left..right]              | invariant   | action
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 = 12 => return 4
m = 12 found: index 3 in the code (position 4 in the pseudo-code)

Check for m = 0, ..., 20: 0 errors


**Try it yourself.** Change the array (it must stay sorted) and the target value.

In [3]:
%%writefile prova_binary_search.cpp
#define SOLO_LIBRERIA   // do not run the main of binary_search.cpp
#include "binary_search.cpp"

int main()
{
    int a[] = {2, 4, 4, 7, 9, 15, 21, 30};      // <-- change the array (sorted!)
    int n = 8;
    int m = 15;                                 // <-- and the target value
    printf("left | right | a[left..right]              | invariant   | action\n");
    int k = BinarySearch(a, n, m, true);
    printf("returns %d\n", k);
    return 0;
}

Writing prova_binary_search.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_binary_search prova_binary_search.cpp && ./prova_binary_search

left | right | a[left..right]              | invariant   | action
   1 |     8 | [2, 4, 4, 7, 9, 15, 21, 30] | ok          | mid = 4, a[4] = 7 < 15 => left = 5
   5 |     8 | [9, 15, 21, 30]             | ok          | mid = 6, a[6] = 15 = 15 => return 6
returns 5


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).